# Crop Recommendation System - Production Implementation
## Principal ML Engineer - Complete Solution

**Project Overview:**
- **Objective**: Build an intelligent AI system for crop recommendation
- **Dataset**: 2200 samples, 7 features, 22 crop classes
- **Approach**: Ensemble methods with hyperparameter optimization
- **Target**: Maximum generalization accuracy with production readiness

**Table of Contents:**
1. Setup and Data Loading
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Model Training and Comparison
5. Hyperparameter Optimization
6. Model Evaluation
7. Feature Importance Analysis
8. Production Inference
9. Results and Conclusions

## 1. Setup and Data Loading

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys
import os

# Add src to path
sys.path.append('../src')

# Import custom modules
from utils import (
    set_random_seeds,
    load_dataset,
    get_feature_statistics,
    calculate_memory_usage,
    PerformanceTimer
)
from preprocessing import (
    prepare_data,
    check_data_quality,
    remove_duplicates,
    detect_outliers_iqr
)
from training import ModelTrainer
from inference import CropRecommendationSystem

# Configure visualization
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Set random seeds for reproducibility
RANDOM_STATE = 42
set_random_seeds(RANDOM_STATE)

print("✓ All libraries imported successfully")
print(f"Python version: {sys.version}")

In [ ]:
# Load dataset
DATA_PATH = '../Crop-recommendation.csv'

with PerformanceTimer("Data Loading"):
    df = load_dataset(DATA_PATH, validate=True)

print(f"\nDataset Shape: {df.shape}")
print(f"Memory Usage: {calculate_memory_usage(df)['total_mb']} MB")
print("\nFirst 5 rows:")
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Basic dataset information
print("=" * 80)
print("DATASET INFORMATION")
print("=" * 80)
print(f"\nNumber of samples: {len(df):,}")
print(f"Number of features: {len(df.columns) - 1}")
print(f"Number of classes: {df['label'].nunique()}")
print(f"\nFeature columns: {', '.join([col for col in df.columns if col != 'label'])}")
print(f"\nTarget column: label")
print(f"\nCrop classes: {sorted(df['label'].unique())}")

# Data types
print("\n" + "=" * 80)
print("DATA TYPES")
print("=" * 80)
print(df.dtypes)

In [ ]:
# Data quality check
quality_report = check_data_quality(df)

print("=" * 80)
print("DATA QUALITY REPORT")
print("=" * 80)
print(f"\nTotal Rows: {quality_report['n_rows']:,}")
print(f"Total Columns: {quality_report['n_columns']}")
print(f"\nMissing Values: {sum(quality_report['missing_values'].values())}")
print(f"Duplicate Rows: {quality_report['duplicate_rows']}")
print(f"Memory Usage: {quality_report['memory_usage_mb']:.2f} MB")

if quality_report['duplicate_rows'] > 0:
    df = remove_duplicates(df)
    print(f"\n✓ Removed {quality_report['duplicate_rows']} duplicate rows")

In [ ]:
# Statistical summary
feature_cols = [col for col in df.columns if col != 'label']
stats = get_feature_statistics(df, feature_cols)

print("=" * 80)
print("FEATURE STATISTICS")
print("=" * 80)
print(stats)

In [ ]:
# Class distribution
class_counts = df['label'].value_counts().sort_index()

print("=" * 80)
print("CLASS DISTRIBUTION")
print("=" * 80)
print(f"\n{class_counts}")
print(f"\nMost common class: {class_counts.idxmax()} ({class_counts.max()} samples)")
print(f"Least common class: {class_counts.idxmin()} ({class_counts.min()} samples)")
print(f"\nClass balance ratio: {class_counts.max() / class_counts.min():.2f}:1")

# Visualize class distribution
fig, ax = plt.subplots(figsize=(14, 6))
class_counts.plot(kind='bar', ax=ax, color='skyblue', edgecolor='black')
ax.set_title('Crop Class Distribution', fontsize=16, fontweight='bold')
ax.set_xlabel('Crop Type', fontsize=12)
ax.set_ylabel('Number of Samples', fontsize=12)
ax.axhline(y=class_counts.mean(), color='red', linestyle='--', label=f'Mean: {class_counts.mean():.0f}')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/figures/class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved: reports/figures/class_distribution.png")

In [ ]:
# Feature distributions
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.ravel()

for idx, col in enumerate(feature_cols):
    axes[idx].hist(df[col], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{col} Distribution', fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(True, alpha=0.3)

# Remove extra subplot
fig.delaxes(axes[-1])

plt.suptitle('Feature Distributions', fontsize=18, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('../reports/figures/feature_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Figure saved: reports/figures/feature_distributions.png")

In [ ]:
# Correlation matrix
correlation_matrix = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=1,
    cbar_kws={"shrink": 0.8},
    ax=ax
)
ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../reports/figures/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Figure saved: reports/figures/correlation_matrix.png")
print("\nKey Correlations (|r| > 0.5):")
# Find strong correlations
strong_corr = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.5:
            strong_corr.append((
                correlation_matrix.columns[i],
                correlation_matrix.columns[j],
                correlation_matrix.iloc[i, j]
            ))

if strong_corr:
    for feat1, feat2, corr in strong_corr:
        print(f"  {feat1} ↔ {feat2}: {corr:.3f}")
else:
    print("  No strong correlations found (all |r| < 0.5)")

In [ ]:
# Outlier detection
outlier_df = detect_outliers_iqr(df, feature_cols, threshold=1.5)

print("=" * 80)
print("OUTLIER ANALYSIS (IQR Method)")
print("=" * 80)
print("\nOutlier counts per feature:")
for col in feature_cols:
    n_outliers = outlier_df[f'{col}_outlier'].sum()
    pct = (n_outliers / len(df)) * 100
    print(f"  {col:15s}: {n_outliers:4d} ({pct:5.2f}%)")

# Visualize outliers
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.ravel()

for idx, col in enumerate(feature_cols):
    axes[idx].boxplot(df[col], vert=True)
    axes[idx].set_title(f'{col} - Outliers', fontweight='bold')
    axes[idx].set_ylabel(col)
    axes[idx].grid(True, alpha=0.3)

fig.delaxes(axes[-1])
plt.suptitle('Outlier Detection (Box Plots)', fontsize=18, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('../reports/figures/outlier_detection.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved: reports/figures/outlier_detection.png")
print("\nNote: Outliers are not removed as they may represent valid crop requirements")